# From print() to structured Python logs in Elastic

Requires Python 3.10+ and Elasticsearch 9.5+ with the Managed OTLP endpoint. Copy `.env.example`
to `.env` and set an API key created in Kibana **Dev Tools**:

```
POST /_security/api_key
{ "name": "structured-python-logs-notebook", "role_descriptors": {} }
```

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

Every query below filters on the start time of this run, so the counts match the article even if
you run the notebook more than once.

In [ ]:
import os
import pathlib
import subprocess
import sys
import time
import warnings
from datetime import datetime, timedelta, timezone
from urllib.parse import quote

from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv()
# ES|QL adds a default LIMIT of 1000 and warns about it; the lab has 56 records.
warnings.filterwarnings("ignore", message="No limit defined")

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"
OTLP_API_KEY = os.getenv("OTLP_API_KEY") or API_KEY

# The two connection variables from the article. The scripts run as child processes
# and inherit them.
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = os.environ["OTLP_ENDPOINT"].rstrip("/")
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=ApiKey {OTLP_API_KEY}"

# Names of everything this notebook creates. Clean up deletes exactly these.
# STREAM is shared with other OTLP logs, so Clean up removes only the records of SERVICES.
STREAM = "logs-generic.otel-default"
SERVICES = ["shipping-text", "shipping-structured"]
FILES = [
    "shipments.py",
    "shipping_print.py",
    "telemetry.py",
    "shipping_text.py",
    "shipping_logger.py",
]

RUN_START = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")

es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)


def esql(query, title=None):
    """Run an ES|QL query and print the result as a text table."""
    response = es.esql.query(query=query, format="txt")
    if title:
        print(f"\n{title}")
    print(response.body)


def run(script):
    """Run one of the lab scripts and return (stdout, stderr)."""
    result = subprocess.run(
        [sys.executable, script], capture_output=True, text=True, check=True
    )
    return result.stdout, result.stderr


def wait_for(service, expected, timeout=120):
    """Wait until this run's records for a service are searchable."""
    query = f"""
FROM {STREAM}
| WHERE service.name == "{service}" AND @timestamp >= "{RUN_START}"
| STATS n = COUNT(*)
"""
    found, stable = 0, 0
    deadline = time.time() + timeout
    while time.time() < deadline and stable < 3:
        try:
            found = es.esql.query(query=query).body["values"][0][0]
        except Exception:
            # the data stream does not exist until the first record arrives
            found = 0
        if found >= expected:
            # Refresh so every shard copy sees the records before the next query runs.
            es.options(ignore_status=[403]).indices.refresh(index=STREAM)
            stable += 1
            time.sleep(1)
        else:
            stable = 0
            time.sleep(3)
    print(f"{service}: {found} records since {RUN_START} (expected {expected})")
    if found != expected:
        raise RuntimeError(
            f"{service}: found {found} records, expected {expected}. "
            "Check OTLP_ENDPOINT and the API key, and the export errors printed by the script."
        )


def discover_link(query):
    """Kibana Discover link for an ES|QL query over this run's time range."""

    def rison(value):
        return "'" + value.replace("!", "!!").replace("'", "!'") + "'"

    end = (datetime.now(timezone.utc) + timedelta(minutes=5)).strftime(
        "%Y-%m-%dT%H:%M:%S.000Z"
    )
    g = f"(filters:!(),refreshInterval:(pause:!t,value:60000),time:(from:{rison(RUN_START)},to:{rison(end)}))"
    a = f"(columns:!(),dataSource:(type:esql),filters:!(),hideChart:!t,interval:auto,query:(esql:{rison(query.strip())}),sort:!())"
    return f"{KBN_BASE}/app/discover#/?_g={quote(g, safe='')}&_a={quote(a, safe='')}"


print(
    f"elasticsearch {es.info()['version']['number']} | space={SPACE} | run start {RUN_START}"
)

## 2. Write the coffee roaster's shipping script

`shipments.py` holds the fixed, simulated responses of the three marketplaces: 12 deliveries,
four per partner, with the HTTP status of every attempt. Two deliveries recover after a retry
(05 and 06) and two ultimately fail (07 and 09). No marketplace API is needed.

In [ ]:
%%writefile shipments.py
# Fixed, simulated marketplace responses:
# (delivery id, partner, HTTP status of each attempt)
shipments = [
    ("delivery-01", "harbor", [204]),
    ("delivery-02", "harbor", [204]),
    ("delivery-03", "harbor", [204]),
    ("delivery-04", "harbor", [204]),
    ("delivery-05", "ridge", [503, 204]),
    ("delivery-06", "ridge", [503, 204]),
    ("delivery-07", "ridge", [503, 503, 503]),
    ("delivery-08", "ridge", [204]),
    ("delivery-09", "mesa", [400]),
    ("delivery-10", "mesa", [204]),
    ("delivery-11", "mesa", [204]),
    ("delivery-12", "mesa", [204]),
]

In [ ]:
%%writefile shipping_print.py
from shipments import shipments

for delivery_id, partner, responses in shipments:
    for attempt, status in enumerate(responses, start=1):
        outcome = "sent" if status < 300 else "failed"
        print(f"Sending {delivery_id} to {partner}: "
              f"attempt {attempt}, HTTP {status}, {outcome}")
    print(f"Finished {delivery_id}: {outcome}")

Run it. The output is readable, but searching it for `failed` also returns deliveries that
later succeeded.

In [ ]:
stdout, _ = run("shipping_print.py")
print(stdout)

lines = stdout.splitlines()
print(f"{len(lines)} lines, {sum('failed' in line for line in lines)} contain 'failed'")

## 3. Send the messages to Elastic

`telemetry.py` connects Python's logger to the OpenTelemetry exporter. `LoggingHandler`
forwards log records to OpenTelemetry and `StreamHandler` also shows them in the terminal.
The exporter reads the two `OTEL_EXPORTER_OTLP_*` variables set in step 1.

In [ ]:
%%writefile telemetry.py
import logging
from opentelemetry.exporter.otlp.proto.http._log_exporter import OTLPLogExporter
from opentelemetry.instrumentation.logging.handler import LoggingHandler
from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import BatchLogRecordProcessor
from opentelemetry.sdk.resources import Resource


def setup_logging(service_name):
    provider = LoggerProvider(
        resource=Resource.create({"service.name": service_name})
    )
    provider.add_log_record_processor(BatchLogRecordProcessor(OTLPLogExporter()))
    logger = logging.getLogger(service_name)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    logger.addHandler(LoggingHandler(logger_provider=provider))
    logger.addHandler(logging.StreamHandler())
    return logger, provider

`shipping_text.py` keeps the original message text and only replaces `print()` with
`logger.info()`. `provider.shutdown()` flushes the batch before the script exits.

In [ ]:
%%writefile shipping_text.py
from shipments import shipments
from telemetry import setup_logging

logger, provider = setup_logging("shipping-text")
for delivery_id, partner, responses in shipments:
    for attempt, status in enumerate(responses, start=1):
        outcome = "sent" if status < 300 else "failed"
        logger.info(f"Sending {delivery_id} to {partner}: "
                    f"attempt {attempt}, HTTP {status}, {outcome}")
    logger.info(f"Finished {delivery_id}: {outcome}")
provider.shutdown()

In [ ]:
_, stderr = run("shipping_text.py")
print(stderr)

# 28 records: 16 attempts and 12 completions
wait_for("shipping-text", 28)

Search the messages for `failed`. The article shows eight matching lines for only two failed
deliveries, because failed attempts and final outcomes are mixed in the text.

In [ ]:
TEXT_FAILED = f"""
FROM {STREAM}
| WHERE service.name == "shipping-text" AND body.text LIKE "*failed*"
  AND @timestamp >= "{RUN_START}"
| KEEP body.text
| SORT body.text
| LIMIT 100
"""
esql(TEXT_FAILED, "Text logs containing 'failed' (expected: 8 rows)")

esql(
    f"""
FROM {STREAM}
| WHERE service.name == "shipping-text" AND body.text LIKE "*failed*"
  AND @timestamp >= "{RUN_START}"
| STATS matching_lines = COUNT(*)
"""
)

## 4. Add fields to the logger

`shipping_logger.py` moves the delivery details into `extra`, which `LoggingHandler` exports
as OpenTelemetry attributes. `delivery.event` distinguishes an attempt from a completion.

In [ ]:
%%writefile shipping_logger.py
from shipments import shipments
from telemetry import setup_logging

logger, provider = setup_logging("shipping-structured")
for delivery_id, partner, responses in shipments:
    for attempt, status in enumerate(responses, start=1):
        fields = {
            "delivery.id": delivery_id,
            "delivery.partner": partner,
            "delivery.attempt": attempt,
            "delivery.outcome": "sent" if status < 300 else "failed",
            "http.response.status_code": status,
        }
        logger.info("Sending shipment update", extra={
            **fields, "delivery.event": "attempt"
        })
    logger.info("Shipment update completed", extra={
        **fields, "delivery.event": "completed"
    })
provider.shutdown()

In [ ]:
_, stderr = run("shipping_logger.py")
print(stderr)

wait_for("shipping-structured", 28)

Both versions send 28 records: 16 attempts and 12 completions. The second query shows one
structured record with its fields, including a numeric status code.

In [ ]:
esql(
    f"""
FROM {STREAM}
| WHERE service.name == "shipping-structured" AND @timestamp >= "{RUN_START}"
| STATS records = COUNT(*) BY delivery.event
| SORT delivery.event
""",
    "Structured records by event (expected: attempt 16, completed 12)",
)

esql(
    f"""
FROM {STREAM}
| WHERE service.name == "shipping-structured" AND @timestamp >= "{RUN_START}"
  AND delivery.id == "delivery-07" AND delivery.event == "completed"
| KEEP body.text, delivery.attempt, delivery.event, delivery.id, delivery.outcome,
       delivery.partner, http.response.status_code
""",
    "One structured record",
)

## 5. Filter failed deliveries

Ask for failed completions instead of messages that contain a word. The article shows two rows:
delivery-07 for ridge with HTTP 503 and delivery-09 for mesa with HTTP 400. Changing
`delivery.event` to `"attempt"` counts failed attempts instead: six in this run.

In [ ]:
FAILED_COMPLETIONS = f"""
FROM {STREAM}
| WHERE service.name == "shipping-structured"
  AND delivery.event == "completed"
  AND delivery.outcome == "failed"
  AND @timestamp >= "{RUN_START}"
| KEEP delivery.id, delivery.partner, http.response.status_code
| SORT delivery.id
| LIMIT 100
"""
esql(FAILED_COMPLETIONS, "Failed deliveries (expected: 2 rows)")

esql(
    f"""
FROM {STREAM}
| WHERE service.name == "shipping-structured"
  AND delivery.event == "attempt"
  AND delivery.outcome == "failed"
  AND @timestamp >= "{RUN_START}"
| STATS failed_attempts = COUNT(*)
""",
    "Failed attempts (expected: 6)",
)

## 6. Ask AI Agent

This step is manual. Open Elastic AI Agent in Kibana and, in a separate chat for each service,
give it the service name, the data stream, and the time range printed below. Then ask:

> Which deliveries ultimately failed, and which recovered after retrying? Include partner and
> final HTTP status. Count failed attempts separately from final outcomes. Show the ES|QL
> queries you used.

> Calculate the final failed-delivery percentage by partner from completions only. Show the
> exact queries and results.

The query below computes the answer the agent should reach: harbor 0%, mesa 25%, ridge 25%.
The Discover links open the article's two queries over the time range of this run.

In [ ]:
now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
print(f"Services: {', '.join(SERVICES)}")
print(f"Data stream: {STREAM}")
print(f"Time range: {RUN_START} to {now}")

esql(
    f"""
FROM {STREAM}
| WHERE service.name == "shipping-structured"
  AND delivery.event == "completed"
  AND @timestamp >= "{RUN_START}"
| STATS total = COUNT(*), failed = COUNT(*) WHERE delivery.outcome == "failed" BY delivery.partner
| EVAL failed_pct = ROUND(100.0 * failed / total, 1)
| SORT delivery.partner
""",
    "Failed-delivery percentage by partner, from completions only",
)

# The article's two queries, as typed in Discover. The time picker carries the range.
print("Discover, text logs containing 'failed':")
print(
    discover_link(
        f"""
FROM {STREAM}
| WHERE service.name == "shipping-text" AND body.text LIKE "*failed*"
| KEEP body.text
| LIMIT 100
"""
    )
)
print("\nDiscover, failed deliveries from fields:")
print(
    discover_link(
        f"""
FROM {STREAM}
| WHERE service.name == "shipping-structured"
  AND delivery.event == "completed"
  AND delivery.outcome == "failed"
| KEEP delivery.id, delivery.partner, http.response.status_code
| SORT delivery.id
| LIMIT 100
"""
    )
)

## 7. Clean up

Deletes only what this notebook created: the log records of the two `shipping-*` services and
the five script files. `logs-generic.otel-default` is shared with other OTLP logs, so the data
stream itself stays.

In [ ]:
response = es.delete_by_query(
    index=STREAM,
    query={"terms": {"resource.attributes.service.name": SERVICES}},
    conflicts="proceed",
    refresh=True,
)
print(f"deleted {response['deleted']} log records")

for name in FILES:
    pathlib.Path(name).unlink(missing_ok=True)
print("done")